In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta

DATA_RUNTIME = Path("data/runtime")
OUTPUT_HOURLY = Path("output/hourly")

for p in (DATA_RUNTIME, OUTPUT_HOURLY):
    p.mkdir(parents=True, exist_ok=True)


In [3]:
today = datetime.now().strftime("%Y%m%d") 

csv_files = sorted(DATA_RUNTIME.glob(f"runtime_suggestions_{today}*.csv"))
recent_files = []

for f in csv_files:
    try:        
        recent_files.append(f)
    except:
        continue

if not recent_files:
    raise RuntimeError("❌ No recent runtime_suggestions files found.")

In [4]:

# Load all recent files
df_list = []
for f in recent_files:
    ts = datetime.strptime(f.stem.split("_")[-1], "%Y%m%d%H%M%S")
    temp = pd.read_csv(f)
    temp["timestamp"] = ts
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
df.sort_values("timestamp", inplace=True)
df.shape

(12, 15)

In [5]:
df_plot = (
    df.sort_values("timestamp")
      .drop_duplicates(subset="symbol", keep="last")
)
df_plot.head()

,symbol,regularMarketPrice,regularMarketChangePercent,regularMarketVolume,averageDailyVolume3Month,marketCap,VolumeSpike,MomentumScore,VolumeScore,VolatilityScore,TrendScore,HotScore_today,HotScore_avg,RuntimeScore,timestamp
0,NTRA,259.97,10.73860,2938702.0,1481398.0,3.723170e+10,1.983736,0.980892,0.927813,0.974522,0.885350,0.951486,0.951486,0.975743,2026-06-25 15:05:23
1,KBH,61.51,16.65090,4563274.0,1254400.0,3.850526e+09,3.637814,1.000000,0.991507,0.887473,0.732484,0.947771,0.947771,0.973885,2026-06-25 15:05:23
2,ICLR,158.17,10.86420,2406383.0,1171858.0,1.211607e+10,2.053477,0.983015,0.929936,0.954352,0.861996,0.946603,0.946603,0.973301,2026-06-25 15:05:23
3,MANE,117.29,13.21430,1039822.0,589237.0,4.900222e+09,1.764692,0.993631,0.900212,0.946921,0.898089,0.942038,0.942038,0.971019,2026-06-25 15:05:23
4,PRIM,92.69,9.11125,5785581.0,1980904.0,5.028890e+09,2.920677,0.963907,0.985138,0.870488,0.728238,0.929087,0.929087,0.964544,2026-06-25 15:05:23


In [6]:
import plotly.express as px


fig = px.scatter(
    df_plot.sort_values(by=["regularMarketPrice"], ascending=True),
    x='RuntimeScore',
    y='HotScore_today',
    size='TrendScore',
    color='HotScore_today',
    text="regularMarketPrice",
    hover_name='symbol',
    log_x=True,
    title='RuntimeScore vs HotScore',
    color_continuous_scale="ice",
)

fig.update_traces(textposition="top right", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top Runtime Score Stocks", 
    xaxis_title="Runtime Score",
    yaxis_title="Hot Score",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=800,
    margin=dict(l=40, r=40, t=60, b=40)
)


fig.show()

chart_path = os.path.join(OUTPUT_HOURLY, "scatter_runtime.html")
fig.write_html(chart_path, include_plotlyjs="cdn")

In [7]:
agg_cols = ["RuntimeScore", "HotScore_today", "TrendScore", "regularMarketPrice"]
grouped = df.groupby("symbol")[agg_cols].mean().sort_values("RuntimeScore", ascending=False).reset_index()
grouped.head()


,symbol,RuntimeScore,HotScore_today,TrendScore,regularMarketPrice
0,NTRA,0.975743,0.951486,0.885350,259.97
1,KBH,0.973885,0.947771,0.732484,61.51
2,ICLR,0.973301,0.946603,0.861996,158.17
3,MANE,0.971019,0.942038,0.898089,117.29
4,PRIM,0.964544,0.929087,0.728238,92.69


In [8]:
fig = px.bar(   
    grouped.sort_values(by=["regularMarketPrice"], ascending=True).head(20), 
    x="RuntimeScore",
    y="symbol", 
    orientation="h",
    color="HotScore_today",
    text="regularMarketPrice",
    hover_data=["TrendScore"], 
    title="Runtime Scores vs TrendScore",
    color_continuous_scale="ice",
)

fig.update_traces(textposition="inside", marker=dict(line=dict(width=1, color="white")))

fig.update_layout(
    title="Top 20 Runtime Score Stocks",
    xaxis_title="Runtime Score",
    yaxis_title="STOCKS",    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=1000,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

chart_path = os.path.join(OUTPUT_HOURLY, "bar_runtime.html")
fig.write_html(chart_path, include_plotlyjs="cdn")